In [1]:
import torch

print("Is CUDA (GPU) available?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("Running on: CPU")

Is CUDA (GPU) available? True
Device Name: NVIDIA GeForce RTX 3060 Laptop GPU


تعديل الكلاسات 

In [4]:
import os

def change_class_id(labels_folder_path, new_id):
    """
    دالة بتدخل جوه فولدر الـ labels وتغير أول رقم (الـ Class ID) للقيمة الجديدة
    """
    # التأكد من أن المسار صحيح
    if not os.path.exists(labels_folder_path):
        print(f"المسار غير موجود: {labels_folder_path}")
        return
        
    count = 0
    # المرور على كل ملفات التكست جوه الفولدر
    for filename in os.listdir(labels_folder_path):
        if filename.endswith('.txt') and filename != 'classes.txt':
            file_path = os.path.join(labels_folder_path, filename)
            
            # 1. قراءة محتوى الملف
            with open(file_path, 'r') as file:
                lines = file.readlines()
            
            # 2. تعديل رقم الكلاس في بداية كل سطر
            new_lines = []
            for line in lines:
                parts = line.split()
                if len(parts) > 0:
                    parts[0] = str(new_id) # تغيير الرقم الأول للـ ID الجديد
                    new_lines.append(" ".join(parts) + "\n")
            
            # 3. كتابة التعديل الجديد في الملف
            with open(file_path, 'w') as file:
                file.writelines(new_lines)
            count += 1
            
    print(f"تم بنجاح تعديل {count} ملف في الفولدر إلى الكلاس رقم: {new_id}")

# =========================================================
# تشغيل الدالة: حط هنا مسارات فولدرات الـ labels الحقيقية عندك
# =========================================================

# مثال للرز (عايزينه يبقى رقم 1):
# كرر السطر ده لكل فولدر (train, valid, test) الخاص بالرز فقط قبل ما تدمجهم
change_class_id(r"D:\gp_pro_sasa\data\rice\augmentation\train\labels", new_id=1)
change_class_id(r"D:\gp_pro_sasa\data\rice\augmentation\valid\labels", new_id=1)
change_class_id(r"D:\gp_pro_sasa\data\rice\augmentation\test\labels", new_id=1)


# مثال للسلطة (عايزينها تبقى رقم 2):
# كرر السطر ده لكل فولدر (train, valid, test) الخاص بالسلطة فقط قبل ما تدمجهم
change_class_id(r"D:\gp_pro_sasa\data\salad\augmentation\Salad.v1-aumentation_salad.yolov8\train\labels", new_id=2)
change_class_id(r"D:\gp_pro_sasa\data\salad\augmentation\Salad.v1-aumentation_salad.yolov8\valid\labels", new_id=2)
change_class_id(r"D:\gp_pro_sasa\data\salad\augmentation\Salad.v1-aumentation_salad.yolov8\test\labels", new_id=2)

تم بنجاح تعديل 294 ملف في الفولدر إلى الكلاس رقم: 1
تم بنجاح تعديل 28 ملف في الفولدر إلى الكلاس رقم: 1
تم بنجاح تعديل 14 ملف في الفولدر إلى الكلاس رقم: 1
تم بنجاح تعديل 75 ملف في الفولدر إلى الكلاس رقم: 2
تم بنجاح تعديل 5 ملف في الفولدر إلى الكلاس رقم: 2
تم بنجاح تعديل 6 ملف في الفولدر إلى الكلاس رقم: 2


In [9]:
import os

def remap_and_filter_labels(labels_folder, mapping_dict):
    """
    سكريبت لتعديل وتنظيف ملفات الـ txt في أي فولدر (Train, Val, Test)
    """
    if not os.path.exists(labels_folder):
        # لو الفولدر مش موجود (زي لو مفيش فولدر test مثلاً)، الكود هيكمل عادي من غير ما يقف
        return
        
    txt_files = [f for f in os.listdir(labels_folder) if f.endswith('.txt')]
    modified_count = 0
    
    for file_name in txt_files:
        file_path = os.path.join(labels_folder, file_name)
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
            
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
                
            old_class_id = int(parts[0])
            
            # تعديل الرقم للجديد الموحد
            if old_class_id in mapping_dict:
                parts[0] = str(mapping_dict[old_class_id])
                new_line = " ".join(parts) + "\n"
                new_lines.append(new_line)
            else:
                # حذف كلاس الـ null تلقائياً
                continue
                
        with open(file_path, 'w') as f:
            f.writelines(new_lines)
        modified_count += 1

    print(f"🎯 تم تعديل وتنظيف {modified_count} ملف بنجاح في: {labels_folder}")

# =====================================================================
# 🔥 تشغيل التعديل الشامل (Train + Val + Test)
# =====================================================================

# 1. داتا سيت البيتزا
pizza_mapping = {0: 4}
print("--- جاري تعديل داتا سيت البيتزا ---")
remap_and_filter_labels(r"D:\gp_pro_sasa\data\pizza\augmentation\train\labels", pizza_mapping)
remap_and_filter_labels(r"D:\gp_pro_sasa\data\pizza\augmentation\valid\labels", pizza_mapping)
remap_and_filter_labels(r"D:\gp_pro_sasa\data\pizza\augmentation\test\labels", pizza_mapping) # تعديل التيست للبيتزا


# 2. داتا سيت البرجر
burger_mapping = {0: 3, 1: 7, 2: 6, 3: 5, 4: 4}
print("\n--- جاري تعديل داتا سيت البرجر ---")
remap_and_filter_labels(r"D:\gp_pro_sasa\data\burgers+fries+cola+ketchup +mustard\augmentation\train\labels", burger_mapping)
remap_and_filter_labels(r"D:\gp_pro_sasa\data\burgers+fries+cola+ketchup +mustard\augmentation\valid\labels", burger_mapping)
remap_and_filter_labels(r"D:\gp_pro_sasa\data\burgers+fries+cola+ketchup +mustard\augmentation\test\labels", burger_mapping) # تعديل التيست للبرجر


# 3. داتا سيت الأكل المصري
egyptian_food_mapping = {0: 8, 1: 9, 2: 10, 3: 11, 4: 15, 6: 13, 7: 1, 8: 14, 9: 12}
print("\n--- جاري تعديل داتا سيت الأكل المصري ---")
remap_and_filter_labels(r"D:\gp_pro_sasa\data\masry\augmentation\train\labels", egyptian_food_mapping)
remap_and_filter_labels(r"D:\gp_pro_sasa\data\masry\augmentation\valid\labels", egyptian_food_mapping)
remap_and_filter_labels(r"D:\gp_pro_sasa\data\masry\augmentation\test\labels", egyptian_food_mapping) # تعديل التيست للمصري

--- جاري تعديل داتا سيت البيتزا ---
🎯 تم تعديل وتنظيف 6229 ملف بنجاح في: D:\gp_pro_sasa\data\pizza\augmentation\train\labels
🎯 تم تعديل وتنظيف 595 ملف بنجاح في: D:\gp_pro_sasa\data\pizza\augmentation\valid\labels
🎯 تم تعديل وتنظيف 296 ملف بنجاح في: D:\gp_pro_sasa\data\pizza\augmentation\test\labels

--- جاري تعديل داتا سيت البرجر ---
🎯 تم تعديل وتنظيف 2111 ملف بنجاح في: D:\gp_pro_sasa\data\burgers+fries+cola+ketchup +mustard\augmentation\train\labels
🎯 تم تعديل وتنظيف 151 ملف بنجاح في: D:\gp_pro_sasa\data\burgers+fries+cola+ketchup +mustard\augmentation\valid\labels
🎯 تم تعديل وتنظيف 150 ملف بنجاح في: D:\gp_pro_sasa\data\burgers+fries+cola+ketchup +mustard\augmentation\test\labels

--- جاري تعديل داتا سيت الأكل المصري ---
🎯 تم تعديل وتنظيف 11895 ملف بنجاح في: D:\gp_pro_sasa\data\masry\augmentation\train\labels
🎯 تم تعديل وتنظيف 324 ملف بنجاح في: D:\gp_pro_sasa\data\masry\augmentation\valid\labels
🎯 تم تعديل وتنظيف 167 ملف بنجاح في: D:\gp_pro_sasa\data\masry\augmentation\test\labels


In [11]:
from ultralytics import YOLO

# 1. تحميل الموديل الأساسي النظيف من الصفر
model = YOLO("yolo26n.pt")

# 2. بدء التدريب الموحد والجديد بالكامل
model.train(
    data=r"C:\gp_pro_sasa\food_dataset\data.yaml",   # مسار ملف الـ yaml على الـ C
    epochs=70,                                        # إجمالي الـ 70 إيبوك كاملين
    imgsz=640,                                        # حجم الصور الثابت
    batch=16,                                         # حجم الـ batch المتوافق مع الـ 3060
    device=0,                                         # تشغيل كارت الشاشة الـ RTX 3060
    workers=4                                         # لتسريع تحميل الصور من البروسيسور
)

Ultralytics 8.4.53  Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\gp_pro_sasa\food_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-6, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

KeyboardInterrupt: 

comlet training


In [ ]:
from ultralytics import YOLO

# 1. شاور على ملف last.pt اللي جواه الـ 4 Epochs بتوعك
# (تأكد من اسم الفولدر الصح جوه runs/detect، لو كان train-4 أو train-5 حطه بالظبط)
model = YOLO(r"C:\gp_pro_sasa\code\runs\detect\train-5\weights\last.pt")

# 2. تكملة التدريب مع إجبار الكارت على الشغل
model.train(
    resume=True,      # هيكمل من أول الـ Epoch 5 أوتوماتيك
    device=0,         # السطر السحري اللي هيشغل الـ GTX 1650 Ti فوراً
    batch=16,         # الـ 1650 Ti هتشيل الـ batch 16 مستريحة فيimgs 640
    workers=2         # تظبيط نقل البيانات على الويندوز عشان الـ CPU ما يهنقش
)

In [13]:
from ultralytics import YOLO

# 1. تحميل الموديل المستخرج من التدريب (أفضل وزن)
model = YOLO(r"C:\gp_pro_sasa\code\runs\detect\train-6\weights\best.pt")

# 2. تشغيل التوقع مع التعديلات الجديدة لحل مشكلة التداخل والكلاس الواحد
results = model.predict(
    source=r"C:\gp_pro_sasa\unseen_data", 
    conf=0.15,         # قللنا حد الثقة من 0.25 لـ 0.15 عشان يلقط اللحمة والسلطة اللي ثقته فيهم قليلة
    iou=0.85,          # ضيفنا دي ورفعناها لـ 0.85 عشان يظهر الصناديق المتداخلة فوق بعض جوة نفس الطبق
    save=True,         
    save_txt=False     
)

print("تمت عملية التوقع بالتعديلات الجديدة! شيك على الصور في runs/detect/predict")


image 1/7 C:\gp_pro_sasa\unseen_data\EuDdtteXYAYTQjH.jpg: 640x640 1 pizza, 38.9ms
image 2/7 C:\gp_pro_sasa\unseen_data\images (1).jpg: 640x448 (no detections), 69.4ms
image 3/7 C:\gp_pro_sasa\unseen_data\images (2).jpg: 640x640 5 burgers, 1 fries, 13.4ms
image 4/7 C:\gp_pro_sasa\unseen_data\images (4).jpg: 640x640 (no detections), 13.8ms
image 5/7 C:\gp_pro_sasa\unseen_data\images.jpg: 640x544 (no detections), 57.2ms
image 6/7 C:\gp_pro_sasa\unseen_data\teest1_meet+rice+salad.jpg: 640x640 1 rice, 1 chicken, 11.9ms
image 7/7 C:\gp_pro_sasa\unseen_data\test2_meet+rice.jpg: 384x640 (no detections), 20.7ms
Speed: 3.2ms preprocess, 32.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\gp_pro_sasa\code\runs\detect\predict-4
تمت عملية التوقع بالتعديلات الجديدة! شيك على الصور في runs/detect/predict


In [ ]:
import os
from collections import Counter
from ultralytics import YOLO

# 1. تحميل الموديل
model = YOLO(r"C:\gp_pro_sasa\code\runs\detect\train-6\weights\best.pt")

# 2. تعديل الاسم في الطبقة الخارجية للموديل
model.names[15] = "konafa"

# 3. تشغيل التوقع (بدون أي باراميترز غريبة عشان ميرميش Error)
results = model.predict(
    source=r"C:\gp_pro_sasa\unseen_data", 
    imgsz=640,         
    conf=0.05,         
    iou=0.85,          
    save=True,         # 🚫 خليها False هنا مؤقتاً لأننا هنسيفها بنفسنا بالاسم الصح حالا تحت!
    verbose=False      
)

print("\n--- نتائج تحليل الصور وتصحيح الكلمات على الصورة ---")

# 4. المرور على نتائج كل صورة ورسم البوكسات بالأسماء الجديدة وحفظها يدويًا
for result in results:
    image_name = os.path.basename(result.path)
    print(f"\n📷 الصورة: {image_name}")
    
    # 🔄 الخدعة الكبرى: إجبار الـ result على استخدام الأسماء المعدلة قبل الرسم
    result.names = model.names
    
    if len(result.boxes) == 0:
        print("   ❌ لم يتم اكتشاف أي نوع من الأطعمة في هذه الصورة.")
        continue

    # 💾 هنا بنخليه يرسم المربعات بالأسماء الجديدة اللي لسه باصينها له حالا
    # الـ plot() دي بتاخد الأسماء من result.names اللي إحنا عدلناها لـ konafa
    annotated_frame = result.plot()
    
    # تحديد مسار الحفظ (جوه نفس الفولدر اللي يولو متعود يحفظ فيه)
    save_dir = r"C:\gp_pro_sasa\code\runs\detect\predict"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, image_name)
    
    # حفظ الصورة المكتوب عليها konafa باستخدام مكتبة Pillow المدمجة في بايثون
    from PIL import Image
    im = Image.fromarray(annotated_frame[..., ::-1]) # تحويل الألوان من BGR لـ RGB
    im.save(save_path)
    print(f"   💾 تم حفظ الصورة بالاسم الصحيح (konafa) في: {save_path}")
        
    # طباعة العداد في الـ Terminal
    detected_classes = result.boxes.cls.int().tolist()
    class_counts = Counter(detected_classes)
    for class_id, count in class_counts.items():
        class_name = result.names[class_id]
        print(f"   🔹 {class_name}: {count}")

print("\n--- تمت العملية بنجاح والصور أصبحت جاهزة بكلمة konafa! ---")

Results saved to C:\gp_pro_sasa\code\runs\detect\predict-18

--- نتائج تحليل الصور وتصحيح الكلمات على الصورة ---

📷 الصورة: -112-_png_jpg.rf.eed63b033e6a1ac0b96ad97271adf38f.jpg
   💾 تم حفظ الصورة بالاسم الصحيح (konafa) في: C:\gp_pro_sasa\code\runs\detect\predict\-112-_png_jpg.rf.eed63b033e6a1ac0b96ad97271adf38f.jpg
   🔹 falafel: 10

📷 الصورة: -_-_-_jpg.rf.2ab8470ce325380e073e49bc3901815e.jpg
   ❌ لم يتم اكتشاف أي نوع من الأطعمة في هذه الصورة.

📷 الصورة: 202256-4-_jpg.rf.cb815042af205c6688318cdc915e8ba7.jpg
   💾 تم حفظ الصورة بالاسم الصحيح (konafa) في: C:\gp_pro_sasa\code\runs\detect\predict\202256-4-_jpg.rf.cb815042af205c6688318cdc915e8ba7.jpg
   🔹 chicken: 1

📷 الصورة: 202270_jpg.rf.4940b2498beb2641595f60e3963446ef.jpg
   💾 تم حفظ الصورة بالاسم الصحيح (konafa) في: C:\gp_pro_sasa\code\runs\detect\predict\202270_jpg.rf.4940b2498beb2641595f60e3963446ef.jpg
   🔹 chicken: 1

📷 الصورة: 22_jpeg_jpg.rf.7dc7240ed8bb72f1ebe166b22c6f60a5.jpg
   💾 تم حفظ الصورة بالاسم الصحيح (konafa) في: C:\gp_p

In [10]:
import os
from ultralytics import YOLO

# 1. تحديد المسار الموحد للأوزان بتاعتك
original_weights_path = r"C:\gp_pro_sasa\code\runs\detect\train-6\weights\best.pt"
new_weights_path = r"C:\gp_pro_sasa\code\runs\detect\train-6\weights\best_fixed_konafa.pt"

# 2. تحميل الموديل من الملف الأصلي
model = YOLO(original_weights_path)

print("الأسماء القديمة جوه الملف الأصلي:", model.model.names)

# 3. تعديل الاسم جوه أوزان الموديل الحقيقية برمجياً
model.model.names[15] = "konafa"

# 4. حفظ الموديل في ملف "جديد تماماً" لحماية الملف القديم
model.save(new_weights_path)

print("\n========================================================")
print(f"✅ تم الحفظ بنجاح في ملف جديد: {new_weights_path}")
print(f"🔒 ملفك القديم آمن تماماً ولم يتغير في: {original_weights_path}")
print("========================================================")

الأسماء القديمة جوه الملف الأصلي: {0: 'meat', 1: 'rice', 2: 'salad', 3: 'burger', 4: 'pizza', 5: 'ketchup', 6: 'fries', 7: 'cola', 8: 'banana', 9: 'chicken', 10: 'cucumber', 11: 'falafel', 12: 'konafa', 13: 'orange', 14: 'tomato', 15: 'mulukhiyah'}

✅ تم الحفظ بنجاح في ملف جديد: C:\gp_pro_sasa\code\runs\detect\train-6\weights\best_fixed_konafa.pt
🔒 ملفك القديم آمن تماماً ولم يتغير في: C:\gp_pro_sasa\code\runs\detect\train-6\weights\best.pt


In [27]:
import os
from collections import Counter
from ultralytics import YOLO

# استخدام الملف الجديد اللي فيه التعديل
model = YOLO(r"C:\gp_pro_sasa\code\runs\detect\train-6\weights\best_fixed_konafa.pt")

results = model.predict(
    source=r"C:\gp_pro_sasa\unseen_data", 
    imgsz=640,         
    conf=0.05,         
    iou=0.85,          
    save=True,         # هيسيف الصور ويكتب عليها konafa علطول
    verbose=False      
)

print("\n--- نتائج تحليل الصور وعدد الكلاسات ---")
for result in results:
    image_name = os.path.basename(result.path)
    print(f"\n📷 الصورة: {image_name}")
    
    if len(result.boxes) == 0:
        print("   ❌ لم يتم اكتشاف أي نوع من الأطعمة في هذه الصورة.")
        continue
        
    detected_classes = result.boxes.cls.int().tolist()
    class_counts = Counter(detected_classes)
    
    for class_id, count in class_counts.items():
        class_name = model.names[class_id]
        print(f"   🔹 {class_name}: {count}")

Results saved to C:\gp_pro_sasa\code\runs\detect\predict-35

--- نتائج تحليل الصور وعدد الكلاسات ---

📷 الصورة: -112-_png_jpg.rf.eed63b033e6a1ac0b96ad97271adf38f.jpg
   🔹 falafel: 10

📷 الصورة: 202256-4-_jpg.rf.cb815042af205c6688318cdc915e8ba7.jpg
   🔹 chicken: 1

📷 الصورة: 202270_jpg.rf.4940b2498beb2641595f60e3963446ef.jpg
   🔹 chicken: 1

📷 الصورة: 22_jpeg_jpg.rf.7dc7240ed8bb72f1ebe166b22c6f60a5.jpg
   🔹 konafa: 1

📷 الصورة: 78_jpg.rf.9dedf9f216ceb0d8499689d150e37f19.jpg
   🔹 rice: 1

📷 الصورة: EuDdtteXYAYTQjH.jpg
   🔹 pizza: 1

📷 الصورة: Gemini_Generated_Image_xv9mbfxv9mbfxv9m.png
   🔹 tomato: 1
   🔹 banana: 1

📷 الصورة: Tulkarm-souq-10-scaled.jpg
   🔹 tomato: 21
   🔹 cucumber: 1

📷 الصورة: Unknown-1_jpeg.rf.71cc7e4181b92070b9ddc10b27bbbf66.jpg
   🔹 salad: 1

📷 الصورة: WhatsApp Image 2026-05-25 at 8.54.40 PM.jpeg
   🔹 cucumber: 1

📷 الصورة: WhatsApp Image 2026-05-25 at 9.35.12 PM.jpeg
   🔹 salad: 1

📷 الصورة: WhatsApp Image 2026-05-25 at 9.36.15 PM.jpeg
   ❌ لم يتم اكتشاف أي نوع من 